# M2 L2: Vector Mathematics and Similarity Search

**Build brute-force KNN from scratch.  Prove it's correct against `numpy.argsort`.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

A **vector** is just a list of numbers, like `[0.3, 0.5, 0.2]`.  A **distance** is a single number that says how different two lists are.  An **embedding** is a vector that an AI made to represent a sentence or an image — similar things have similar vectors.

**KNN** (K-Nearest Neighbors) means: *show me the K things most similar to this one.*  Every production vector search engine (Pinecone, Qdrant, Weaviate) is a really fast, slightly-imperfect version of KNN.  Today we build the perfect version.  Tomorrow we speed it up.

## The 5 things to know

1. **L2** is the default distance, but **cosine** is the default for text.
2. **`cos(a, b) == inner_product(norm(a), norm(b))`** — pre-normalize and inner product = cosine, 12× faster.
3. **Brute force KNN is O(N*d)** — fine for 100K vectors, dead at 100M.
4. **Recall@K vs brute force** is how you validate any approximate index.
5. **The identity every vector DB uses** — that's why ingest normalizes.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm02-vector-math' / 'L2-vector-math-knn.py'
_spec = importlib.util.spec_from_file_location('L2', _p)
_L2 = importlib.util.module_from_spec(_spec)
sys.modules['L2'] = _L2
_spec.loader.exec_module(_L2)

print(f'Setup OK. Lesson module: {_L2.LESSON_NUMBER}')
print(f'Root: {ROOT}')

Setup OK. Lesson module: M2.L2
Root: /Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-course-public/course/ai-native-de


In [2]:
# Run the 10 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L2, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 31 tests...

  PASS  test_build_random_corpus_is_deterministic_per_seed
  PASS  test_build_sample_index_is_singleton
  PASS  test_cosine_identical
  PASS  test_cosine_similarity_in_bounds_after_projection
  PASS  test_cosine_via_inner_product_matches_direct
  PASS  test_default_query_vector_hash_changes_with_seed
  PASS  test_default_query_vector_hash_is_deterministic
  PASS  test_default_query_vector_is_deterministic_per_seed
  PASS  test_default_query_vector_is_unit_norm
  PASS  test_default_query_vector_takes_dim_required
  PASS  test_inner_product_orthogonal
  PASS  test_knn_cost_usd_per_month_independent_of_latency
  PASS  test_knn_cost_usd_per_month_rejects_negative_inputs
  PASS  test_knn_cost_usd_per_month_scales_linearly
  PASS  test_knn_cost_usd_per_month_zero_queries_is_zero
  PASS  test_knn_latency_ms_returns_stats
  PASS  test_knn_latency_ms_within_budget
  PASS  test_knn_p95_under_budget_benchmark_is_deterministic_per_seed
  PASS  test_knn_p95_under_budget_benchma

  PASS  test_run_demo_seed_42_runs_cleanly
  PASS  test_tiny_index_returns_ids

31/31 tests pass, 0 fail


In [3]:
# === Section 1: distance primitives ===
import numpy as np
a = np.array([1.0, 0.0, 0.0])
b = np.array([0.0, 1.0, 0.0])
c = np.array([1.0, 0.0, 0.0])
print(f'L1(a, b)    = {_L2.l1_distance(a, b):.3f}')
print(f'L2(a, b)    = {_L2.l2_distance(a, b):.3f}')
print(f'L2(a, c)    = {_L2.l2_distance(a, c):.3f}  (identical -> 0)')
print(f'cos(a, b)   = {_L2.cosine_similarity(a, b):.3f}')
print(f'cos(a, c)   = {_L2.cosine_similarity(a, c):.3f}  (identical -> 1)')

L1(a, b)    = 2.000
L2(a, b)    = 1.414
L2(a, c)    = 0.000  (identical -> 0)
cos(a, b)   = 0.000
cos(a, c)   = 1.000  (identical -> 1)


In [4]:
# === Section 2: identity: cos == inner(norm, norm) ===
a = np.array([3.0, 1.0, 0.0])
b = np.array([0.0, 2.0, 1.0])
direct = _L2.cosine_similarity(a, b)
via_ip  = _L2.cos_via_inner_product(a, b)
print(f'direct cos        = {direct:.5f}')
print(f'cos via inner(p)  = {via_ip:.5f}')
print(f'match within 1e-5 : {abs(direct - via_ip) < 1e-5}')

direct cos        = 0.28284
cos via inner(p)  = 0.28284
match within 1e-5 : True


In [5]:
# === Section 3: KNN on the 10-doc sample ===
idx = _L2.build_sample_index()
q = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
               0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)
print('Query is HR-shaped; expecting HR docs on top:\n')
for id_, dist in idx.search(q, k=5):
    print(f'  {id_:<8}  dist={dist:.4f}')

Query is HR-shaped; expecting HR docs on top:

  hr-001    dist=0.0050
  hr-004    dist=0.0050
  hr-002    dist=0.0121
  hr-003    dist=0.0369
  fin-003   dist=0.9005


In [6]:
# === Section 4: metric sweep on 200 unit vectors ===
import time
big = _L2.build_random_corpus()
print(f'Corpus: {big.vectors.shape[0]} vectors of dim {big.vectors.shape[1]}\n')
print(f'{"metric":<10} {"avg ms/query":>14}')
print('-' * 26)
for metric in ('cosine', 'l2', 'l1', 'ip'):
    big.metric = metric
    t0 = time.perf_counter()
    for _ in range(20):
        _ = big.search(big.vectors[0], k=10)
    dt = (time.perf_counter() - t0) / 20
    print(f'{metric:<10} {dt*1000:>12.2f}')

Corpus: 200 vectors of dim 16

metric       avg ms/query
--------------------------
cosine             2.73
l2                 1.14
l1                 0.98
ip                 0.23


In [7]:
# === Section 5: KNN recall@10 vs numpy.argsort ===
q = big.vectors[7]
truth = set(np.argsort([1 - _L2.cosine_similarity(q, v) for v in big.vectors])[:10].tolist())
pred  = set(n.index for n in _L2.knn(q, big.vectors, k=10, metric='cosine'))
print(f'truth size     : {len(truth)}')
print(f'pred size      : {len(pred)}')
print(f'intersection   : {len(truth & pred)}/10')
print(f'recall@10      : {len(truth & pred)/10:.2f}')

truth size     : 10
pred size      : 10
intersection   : 10/10
recall@10      : 1.00


In [8]:
# === Section 6: vectorized KNN (one matmul, not a Python loop) ===
def knn_vectorized(query, matrix, k=10):
    # Assume pre-normalized; one matmul + argsort.
    scores = matrix @ query          # (N,)
    top_k = np.argsort(-scores)[:k]  # descending
    return [(int(i), float(scores[i])) for i in top_k]

q = big.vectors[7]
loop_hits  = _L2.knn(q, big.vectors, k=10, metric='ip')
vec_hits   = knn_vectorized(q, big.vectors, k=10)
loop_ids   = [h.index for h in loop_hits]
vec_ids    = [i for i, _ in vec_hits]
print(f'loop KNN ids         : {loop_ids}')
print(f'vectorized KNN ids   : {vec_ids}')
print(f'match                : {loop_ids == vec_ids}')

# Time both
t0 = time.perf_counter()
for _ in range(100): _L2.knn(q, big.vectors, k=10, metric='ip')
loop_ms = (time.perf_counter() - t0) / 100 * 1000
t0 = time.perf_counter()
for _ in range(100): knn_vectorized(q, big.vectors, k=10)
vec_ms = (time.perf_counter() - t0) / 100 * 1000
print(f'\nloop       : {loop_ms:.3f} ms/query')
print(f'vectorized : {vec_ms:.3f} ms/query')
print(f'speedup    : {loop_ms/vec_ms:.1f}x')

loop KNN ids         : [7, 3, 77, 30, 159, 49, 163, 54, 103, 187]
vectorized KNN ids   : [7, 3, 77, 30, 159, 49, 163, 54, 103, 187]
match                : True

loop       : 0.253 ms/query
vectorized : 0.012 ms/query
speedup    : 20.4x


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[query vector] --> B[Brute-force KNN<br/>O(N*d)]
    C[(N vectors<br/>in RAM)] --> B
    B --> D[sorted top-K]
    D --> E[production swap:<br/>FAISS / HNSW / Qdrant]
    E -.approx.-> F[top-K in O(log N)]
```

**Read as:** Start with brute force as ground truth.  Replace with ANN when latency bites.  Always measure recall@K of the ANN against the brute force.

## Reflect (10 min)

1. Why is cosine the default for text embeddings, not L2?
2. Explain the identity `cos(a, b) == inner_product(norm(a), norm(b))` in your own words.  Why does it matter for production?
3. The vectorized KNN is 5-10× faster than the loop.  What did NumPy do under the hood?
4. Re-run Section 4 with `k=50` instead of `k=10`.  Does the relative ordering of metrics change?
5. (Stretch) Add a new metric `hamming` for binary vectors, with a test.

---

**Read next:**
- `lessons/m03-embeddings/L3-embeddings-and-models.ipynb` — real embeddings, model selection, dual-index migration
- `lessons/m04-ann/L4-ann-internals.ipynb` — HNSW, IVF-PQ, FAISS benchmarks
- `SYLLABUS.md` — the 18 modules and 21-day plan